In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 17 — Load final model for validation
# ─────────────────────────────────────────────────────────────

import torch
import sentencepiece as spm
from transformers import LlamaForCausalLM, LlamaConfig

FINAL_MODEL_DIR = "./BanglaLLM/final_model"
SP_MODEL_PATH   = "./BanglaLLM/tokenizer/bangla_spm.model"   # ← hardcoded
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
# ── Re-declare paths (run this if starting a fresh kernel) ───
TOKENIZER_OUTPUT = "./BanglaLLM/tokenizer"
FINAL_MODEL_DIR  = "./BanglaLLM/final_model"
CLEANED_OUTPUT   = "./BanglaLLM/cleaned"
FINAL_OUTPUT     = "./BanglaLLM/final_dataset"
CHUNK_FILE       = f"{FINAL_OUTPUT}/chunks.arrow"
MAX_LENGTH       = 2048

# Load SP tokenizer
sp = spm.SentencePieceProcessor(model_file=SP_MODEL_PATH)

# Load trained model
config = LlamaConfig.from_pretrained(FINAL_MODEL_DIR)
model  = LlamaForCausalLM.from_pretrained(
    FINAL_MODEL_DIR,
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)
model.to(DEVICE)
model.eval()

print(f"Model loaded from  : {FINAL_MODEL_DIR}")
print(f"Device             : {DEVICE}")
print(f"Parameters         : {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Vocab size         : {sp.get_piece_size():,}")

/home/nafim/anaconda3/envs/bangla_research_312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 75/75 [00:00<00:00, 5743.21it/s]

Model loaded from  : ./BanglaLLM/final_model
Device             : cuda
Parameters         : 51.7M
Vocab size         : 32,000


In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 18 — Perplexity on eval set
# Lower = better. Untrained model: ~32000 (random)
# Good Bengali model: < 50
# ─────────────────────────────────────────────────────────────
# ── Re-build eval_torch (run this if starting a fresh kernel) ─
import pyarrow as pa
import pyarrow.ipc as ipc
import torch
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset

CHUNK_FILE = "./BanglaLLM/final_dataset/chunks.arrow"
MAX_LENGTH = 2048

print("Loading chunked dataset...")
source          = pa.memory_map(CHUNK_FILE, "r")
table           = ipc.open_file(source).read_all()
chunked_dataset = Dataset(table)

split         = chunked_dataset.train_test_split(test_size=0.05, seed=42)
eval_dataset  = split["test"]

class BengaliCLMDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ids = torch.tensor(self.data[idx]["input_ids"], dtype=torch.long)
        return {"input_ids": ids, "labels": ids.clone()}

eval_torch = BengaliCLMDataset(eval_dataset)

print(f"Eval chunks ready : {len(eval_torch):,}")
print(f"Sample shape      : {eval_torch[0]['input_ids'].shape}")

import math
import torch
from torch.utils.data import DataLoader

NUM_EVAL_BATCHES = 100     # how many batches to evaluate (None = full eval set)

eval_loader = DataLoader(
    eval_torch,
    batch_size  = 4,
    shuffle     = False,
    pin_memory  = True,
)

total_loss  = 0.0
total_steps = 0

print("Computing perplexity on eval set...")

with torch.no_grad():
    for i, batch in enumerate(eval_loader):
        if NUM_EVAL_BATCHES and i >= NUM_EVAL_BATCHES:
            break

        input_ids = batch["input_ids"].to(DEVICE)
        labels    = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, labels=labels)
        total_loss  += outputs.loss.item()
        total_steps += 1

        if (i + 1) % 20 == 0:
            running_ppl = math.exp(total_loss / total_steps)
            print(f"  Batch {i+1:>4} | running perplexity: {running_ppl:.2f}", end="\r")

avg_loss    = total_loss / total_steps
perplexity  = math.exp(avg_loss)

print(f"\n{'─' * 50}")
print(f"Batches evaluated : {total_steps:,}")
print(f"Average loss      : {avg_loss:.4f}")
print(f"Perplexity        : {perplexity:.2f}")
print(f"{'─' * 50}")

if perplexity < 20:
    print("Excellent — model has learned Bengali well")
elif perplexity < 50:
    print("Good — model shows solid Bengali understanding")
elif perplexity < 100:
    print("Moderate — model is learning but needs more data/epochs")
else:
    print("High perplexity — model needs more training")

Loading chunked dataset...
Eval chunks ready : 476
Sample shape      : torch.Size([2048])
Computing perplexity on eval set...
  Batch  100 | running perplexity: 351.42
──────────────────────────────────────────────────
Batches evaluated : 100
Average loss      : 5.8620
Perplexity        : 351.42
──────────────────────────────────────────────────
High perplexity — model needs more training


In [3]:
# ─────────────────────────────────────────────────────────────
# CELL 18b — Entropy analysis
# Perplexity = exp(entropy). Entropy breaks down WHERE and
# WHY the model is uncertain, token by token.
# ─────────────────────────────────────────────────────────────

import torch
import torch.nn.functional as F
import math
from torch.utils.data import DataLoader

NUM_ENTROPY_BATCHES = 50   # batches to sample (keep low, it's detailed)

eval_loader = DataLoader(eval_torch, batch_size=1, shuffle=True)

all_entropies   = []   # per-token entropy across all samples
high_ent_tokens = []   # tokens where model is most confused
low_ent_tokens  = []   # tokens where model is most confident

print("Computing per-token entropy...")
print("─" * 55)

model.eval()
with torch.no_grad():
    for i, batch in enumerate(eval_loader):
        if i >= NUM_ENTROPY_BATCHES:
            break

        input_ids = batch["input_ids"].to(DEVICE)          # (1, seq_len)
        logits    = model(input_ids).logits                 # (1, seq_len, vocab)
        logits    = logits[0, :-1, :]                       # drop last (no next token)
        targets   = input_ids[0, 1:]                        # shift right

        # Per-token probability distributions
        probs     = F.softmax(logits, dim=-1)               # (seq_len, vocab)

        # Shannon entropy per token: H = -sum(p * log2(p))
        log_probs = F.log_softmax(logits, dim=-1)
        entropy   = -(probs * log_probs / math.log(2)).sum(dim=-1)  # bits

        all_entropies.extend(entropy.cpu().tolist())

        # Collect high/low entropy token examples
        for pos, (ent, tid) in enumerate(zip(entropy.tolist(), targets.tolist())):
            piece = sp.id_to_piece(tid)
            if ent > 8.0:
                high_ent_tokens.append((ent, piece))
            elif ent < 1.0:
                low_ent_tokens.append((ent, piece))

        if (i + 1) % 10 == 0:
            print(f"  Batch {i+1:>3}/{NUM_ENTROPY_BATCHES}", end="\r")

# ── Aggregate stats ───────────────────────────────────────────
all_entropies.sort()
n           = len(all_entropies)
mean_ent    = sum(all_entropies) / n
median_ent  = all_entropies[n // 2]
p10         = all_entropies[int(n * 0.10)]   # bottom 10% = confident
p90         = all_entropies[int(n * 0.90)]   # top 10%    = uncertain

# Bucket distribution
buckets = {"0–2 bits (confident)": 0, "2–5 bits": 0,
           "5–8 bits": 0, "8+ bits (uncertain)": 0}
for e in all_entropies:
    if   e < 2: buckets["0–2 bits (confident)"] += 1
    elif e < 5: buckets["2–5 bits"] += 1
    elif e < 8: buckets["5–8 bits"] += 1
    else:       buckets["8+ bits (uncertain)"] += 1

print(f"\n{'═' * 55}")
print(f"  PER-TOKEN ENTROPY REPORT  ({n:,} tokens)")
print(f"{'═' * 55}")
print(f"  Mean entropy   : {mean_ent:.3f} bits")
print(f"  Median entropy : {median_ent:.3f} bits")
print(f"  P10 (confident): {p10:.3f} bits")
print(f"  P90 (uncertain): {p90:.3f} bits")
print(f"  Max bits (vocab): {math.log2(sp.get_piece_size()):.1f} bits  ← random baseline")
print(f"\n  Distribution:")
for label, count in buckets.items():
    pct = count / n * 100
    bar = "█" * int(pct / 2)
    print(f"    {label:<26} {pct:5.1f}%  {bar}")

# ── Token-level examples ─────────────────────────────────────
print(f"\n  Most CONFIDENT tokens (entropy < 1 bit):")
confident_sample = sorted(low_ent_tokens, key=lambda x: x[0])[:10]
for ent, piece in confident_sample:
    print(f"    '{piece}'  →  {ent:.3f} bits")

print(f"\n  Most UNCERTAIN tokens (entropy > 8 bits):")
uncertain_sample = sorted(high_ent_tokens, key=lambda x: -x[0])[:10]
for ent, piece in uncertain_sample:
    print(f"    '{piece}'  →  {ent:.3f} bits")

# ── Interpretation ────────────────────────────────────────────
max_bits = math.log2(sp.get_piece_size())
print(f"\n{'─' * 55}")
print(f"  Interpretation:")
print(f"    Random model would score   : {max_bits:.1f} bits")
print(f"    Your model scores          : {mean_ent:.2f} bits")
print(f"    Reduction from random      : {max_bits - mean_ent:.2f} bits  ← what was learned")
if mean_ent < 3:
    print(f"    ✓ Excellent — model is highly confident on Bengali")
elif mean_ent < 6:
    print(f"    ✓ Good — model shows solid Bengali understanding")
elif mean_ent < 10:
    print(f"    ~ Moderate — more data or epochs would help")
else:
    print(f"    ✗ High entropy — model is close to random guessing")
print(f"{'═' * 55}")

Computing per-token entropy...
───────────────────────────────────────────────────────
  Batch  50/50
═══════════════════════════════════════════════════════
  PER-TOKEN ENTROPY REPORT  (102,350 tokens)
═══════════════════════════════════════════════════════
  Mean entropy   : 9.127 bits
  Median entropy : 10.062 bits
  P10 (confident): 4.750 bits
  P90 (uncertain): 11.750 bits
  Max bits (vocab): 15.0 bits  ← random baseline

  Distribution:
    0–2 bits (confident)         2.9%  █
    2–5 bits                     7.9%  ███
    5–8 bits                    14.9%  ███████
    8+ bits (uncertain)         74.3%  █████████████████████████████████████

  Most CONFIDENT tokens (entropy < 1 bit):
    '<s>'  →  0.075 bits
    '<s>'  →  0.075 bits
    '<s>'  →  0.077 bits
    '<s>'  →  0.077 bits
    '<s>'  →  0.077 bits
    '<s>'  →  0.077 bits
    '<s>'  →  0.078 bits
    '<s>'  →  0.078 bits
    '<s>'  →  0.078 bits
    '<s>'  →  0.079 bits

  Most UNCERTAIN tokens (entropy > 8 bits):
    '▁

In [4]:
# ─────────────────────────────────────────────────────────────
# CELL 19 — Text generation quality check
# Tests whether model produces coherent Bengali continuations
# ─────────────────────────────────────────────────────────────

def generate(prompt, max_new_tokens=80, temperature=0.8, top_p=0.9, top_k=50):
    ids       = [sp.bos_id()] + sp.encode(prompt)
    input_ids = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens = max_new_tokens,
            do_sample      = True,
            temperature    = temperature,
            top_p          = top_p,
            top_k          = top_k,
            eos_token_id   = sp.eos_id(),
            pad_token_id   = 0,
            repetition_penalty = 1.2,   # penalise repeated tokens
        )

    # Decode only the newly generated tokens
    new_ids   = output[0][len(ids):].tolist()
    generated = sp.decode(new_ids)
    return generated


# Test prompts — covers different domains in your training data
test_prompts = [
    "বাংলাদেশের ইতিহাস",
    "আমার প্রিয় খাবার হলো",
    "বিজ্ঞান ও প্রযুক্তির",
    "ঢাকা শহরে",
    "শিক্ষার গুরুত্ব হলো",
]

print("=" * 65)
print("GENERATION QUALITY TEST")
print("=" * 65)

for prompt in test_prompts:
    generated = generate(prompt)
    print(f"\nPrompt    : {prompt}")
    print(f"Generated : {generated}")
    print("─" * 65)

GENERATION QUALITY TEST

Prompt    : বাংলাদেশের ইতিহাস
Generated : , অনেক একটা মানুষ এখন এক কথা, এই দেশের মানুষের দেশে যেতে হবে। আমাদেরই সবাই আমার সব থেকে অনেক কম রয়েছে। এখন আমি মনে করি যে, আমরা আর ভালো। আমাদের খুব একটা ব্যাপার ছিল না, তাদের জন্য ভাল। কিন্তু কিছু হতে পারে না? এমন কোনো রাজনৈতিক কারণে কাজ করার প্রয়োজন নেই। আর কেউ জানে না কেন, এটাই কি সম্ভব নয় সেটা যেন ভালো থাকবে। তবে এ বিষয়ে আপনারা
─────────────────────────────────────────────────────────────────

Prompt    : আমার প্রিয় খাবার হলো
Generated : । এর জন্য আমি আপনার বই ও কিছু করে নিয়ে কাজ করেছিলাম, কিন্তু মনে হয় না। যদি আপনি এ থেকে কি হবে? আমরা অনেক ভাল আছি, আপনি কী জানতে পারেন! আমার সাথে যখন আমাদের কাজের প্রতি দরকার হচ্ছে। এখন এই ধরণের কোন সুযোগ নেই। আমরা তো আমরা বলেছি তারা একটা গুরুত্বপূর্ণ তথ্য দেখতে পাবেন এবং এটি আপনাকে খুব কঠিন নয়। কারণ আপনার কাছে আপনার নিজের জীবনেই হতে পারে। তাই যে
─────────────────────────────────────────────────────────────────

Prompt    : বিজ্ঞান ও প্রযুক্তির
Generated : আওতায় পাঠানোর জন্য 

In [5]:
# ─────────────────────────────────────────────────────────────
# CELL 20 — Tokenizer coverage check
# Verifies SP tokenizer handles Bengali properly
# ─────────────────────────────────────────────────────────────

test_texts = [
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",   # standard sentence
    "বাংলাদেশ স্বাধীন হয়েছিল ১৯৭১ সালে।",       # with digits
    "ক্ষমা করবেন, আপনি কি বাংলায় কথা বলেন?",     # conjuncts (যুক্তাক্ষর)
    "তাপমাত্রা ছিল ৩৭.৫ ডিগ্রি সেলসিয়াস।",      # decimal numbers
]

print("TOKENIZER COVERAGE CHECK")
print("=" * 65)

for text in test_texts:
    tokens   = sp.encode(text, out_type=str)
    ids      = sp.encode(text)
    decoded  = sp.decode(ids)
    is_lossless = (decoded.strip() == text.strip())

    print(f"\nText     : {text}")
    print(f"Tokens   : {tokens}")
    print(f"Count    : {len(tokens)} tokens for {len(text)} chars  "
          f"({len(text)/len(tokens):.1f} chars/token)")
    print(f"Lossless : {'✓' if is_lossless else '✗ MISMATCH — check training data'}")

print("\n" + "=" * 65)
print("Chars/token > 2.0 is good for Bengali (LLaMA default was ~0.8)")

TOKENIZER COVERAGE CHECK

Text     : আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।
Tokens   : ['▁আমার', '▁সোনার', '▁বাংলা', ',', '▁আমি', '▁তোমায়', '▁ভালোবাসি', '।']
Count    : 8 tokens for 38 chars  (4.8 chars/token)
Lossless : ✓

Text     : বাংলাদেশ স্বাধীন হয়েছিল ১৯৭১ সালে।
Tokens   : ['▁বাংলাদেশ', '▁স্বাধীন', '▁হয়েছিল', '▁১৯৭১', '▁সালে', '।']
Count    : 6 tokens for 35 chars  (5.8 chars/token)
Lossless : ✓

Text     : ক্ষমা করবেন, আপনি কি বাংলায় কথা বলেন?
Tokens   : ['▁ক্ষমা', '▁করবেন', ',', '▁আপনি', '▁কি', '▁বাংলায়', '▁কথা', '▁বলেন', '?']
Count    : 9 tokens for 38 chars  (4.2 chars/token)
Lossless : ✓

Text     : তাপমাত্রা ছিল ৩৭.৫ ডিগ্রি সেলসিয়াস।
Tokens   : ['▁তাপমাত্রা', '▁ছিল', '▁৩৭', '.', '৫', '▁ডিগ্রি', '▁সেলসিয়াস', '।']
Count    : 8 tokens for 36 chars  (4.5 chars/token)
Lossless : ✓

Chars/token > 2.0 is good for Bengali (LLaMA default was ~0.8)


In [6]:
# ─────────────────────────────────────────────────────────────
# CELL 21 — Summary report
# ─────────────────────────────────────────────────────────────

print("=" * 65)
print("BANGLA LLM VALIDATION SUMMARY")
print("=" * 65)
print(f"Model dir      : {FINAL_MODEL_DIR}")
print(f"Parameters     : {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Vocab size     : {sp.get_piece_size():,}")
print(f"Perplexity     : {perplexity:.2f}  (lower is better)")
print(f"Avg loss       : {avg_loss:.4f}")
print(f"Eval batches   : {total_steps:,}")
print("=" * 65)

# Guidance for next steps
print("\nNext steps:")
if perplexity < 50:
    print("  → Model is ready. Consider scaling up to 300M params on A100.")
elif perplexity < 100:
    print("  → Run 2-3 more epochs or add more training data.")
else:
    print("  → Perplexity is high. Check data quality and increase training.")

BANGLA LLM VALIDATION SUMMARY
Model dir      : ./BanglaLLM/final_model
Parameters     : 51.7M
Vocab size     : 32,000
Perplexity     : 351.42  (lower is better)
Avg loss       : 5.8620
Eval batches   : 100

Next steps:
  → Perplexity is high. Check data quality and increase training.
